# 06. Model Fit and Residual Diagnostics

Regression is more than fitting a line. We also need to ask whether the line is useful and whether the assumptions behind the inference are plausible.

This chapter covers:

- coefficient of determination $R^2$,
- residual standard error and MSE,
- correlation in simple linear regression,
- the four main residual assumptions.

The assumptions are:

- linear form,
- roughly constant variance,
- roughly normal residuals when inference is needed,
- no obvious time-order pattern.

## Fit the model and collect residuals

## Browser package setup

Run the next cell before the first analysis cell. It loads the scientific Python packages used in this module into the browser-based Python kernel.

The first run in a browser session can take a little time. Later notebooks usually load faster because the browser can reuse cached packages.


In [ ]:
from lite_setup import ensure_packages
await ensure_packages()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import shapiro, anderson
from statsmodels.formula.api import ols

from checks import check, check_close

fuel = pd.read_csv("data/fuelcons.csv")
model = ols("Fuelcons ~ Temp", data=fuel).fit()

diagnostics = fuel.copy()
diagnostics["fitted"] = model.fittedvalues
diagnostics["residual"] = model.resid
diagnostics

## How good is the regression?

A useful simple regression model usually has:

- a small residual scale for the problem context,
- a large enough $R^2$ to be useful,
- a meaningful slope,
- residual plots that do not reveal major assumption problems.

The coefficient of determination is:

$$
R^2 = \frac{SSR}{SST} = 1 - \frac{SSE}{SST}.
$$

It measures the proportion of total response variation explained by the regression model.

In [ ]:
sse = model.ssr
ssr = model.ess
sst = model.centered_tss
mse = model.mse_resid
residual_standard_error = mse ** 0.5
r_squared = model.rsquared
correlation = fuel["Temp"].corr(fuel["Fuelcons"])

print(f"SST = {sst:.3f}")
print(f"SSR = {ssr:.3f}")
print(f"SSE = {sse:.3f}")
print(f"MSE = {mse:.3f}")
print(f"residual standard error = {residual_standard_error:.3f}")
print(f"R-squared = {r_squared:.3f}")
print(f"correlation = {correlation:.3f}")

In [ ]:
print(check_close("R-squared", r_squared, expected=0.899, tolerance=0.005))
print(check(
    correlation < 0,
    "The correlation is negative, matching the negative fitted slope.",
    "Recheck the variables. The relationship in this example should be negative."
))

In this example, $R^2 \approx 0.899$, so the model explains about 89.9% of the total variation in weekly fuel consumption. That sounds strong, but $R^2$ is not a complete model check. A model can have a high $R^2$ and still be inappropriate if the pattern is nonlinear, the variance changes, or time-order dependence is present.

## Residuals sum to approximately zero

In ordinary least squares with an intercept, residuals should sum to approximately zero. This is a useful quick check that you are looking at the model residuals.

In [ ]:
residual_mean = diagnostics["residual"].mean()
residual_sum = diagnostics["residual"].sum()

print(f"Mean residual: {residual_mean:.6f}")
print(f"Sum of residuals: {residual_sum:.6f}")

print(check_close("mean residual", residual_mean, expected=0.0, tolerance=1e-10))

## Residuals vs fitted values

This plot is a first check for nonlinear pattern and unequal variance.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(diagnostics["fitted"], diagnostics["residual"])
ax.axhline(0, color="red", linestyle="--")
ax.set_xlabel("Fitted value")
ax.set_ylabel("Residual")
ax.set_title("Residuals vs fitted values")
plt.show()

What to look for:

- Good sign: residuals are scattered around zero without a strong curve.
- Warning sign: a U-shape, inverted U-shape, or funnel shape.
- In this tiny data set, do not overinterpret minor patterns.

## Normal Q-Q plot

The Q-Q plot checks whether residuals are roughly compatible with a normal distribution.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
stats.probplot(diagnostics["residual"], plot=ax)
ax.set_title("Normal Q-Q plot of residuals")
plt.show()

## Histogram

With only 8 observations, a histogram is rough. It is still useful as a habit.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(diagnostics["residual"], bins=5, edgecolor="black")
ax.set_xlabel("Residual")
ax.set_ylabel("Count")
ax.set_title("Residual histogram")
plt.show()

## Residuals vs observation order

Because the rows are weeks, observation order matters. A time-order pattern can indicate autocorrelation or omitted time structure.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(diagnostics["week"], diagnostics["residual"], marker="o")
ax.axhline(0, color="red", linestyle="--")
ax.set_xlabel("Week")
ax.set_ylabel("Residual")
ax.set_title("Residuals by week")
plt.show()

## Formal normality tests

Formal tests are not magic. With small samples, they have low power; with huge samples, they may flag unimportant deviations. Use them as supporting evidence, not as a substitute for plots and context.

In [ ]:
shapiro_result = shapiro(diagnostics["residual"])
anderson_result = anderson(diagnostics["residual"], method="interpolate")

print("Shapiro-Wilk statistic:", round(shapiro_result.statistic, 4))
print("Shapiro-Wilk p-value:", round(shapiro_result.pvalue, 4))
print()
print("Anderson-Darling statistic:", round(anderson_result.statistic, 4))
print("Anderson-Darling p-value:", round(anderson_result.pvalue, 4))

## Checkpoint: write a diagnostic conclusion

Write a short diagnostic conclusion in your own notes. A useful conclusion should mention at least two diagnostic plots and say whether you would be comfortable using the model for a small interpolation, such as predicting near `Temp = 40`.

Use this structure:

1. What do the residuals vs fitted plot and residuals by week plot suggest?
2. What do the Q-Q plot and histogram suggest, given the small sample size?
3. Would you use this model for interpolation inside the observed temperature range? What uncertainty would you report?

```{admonition} Reference diagnostic conclusion
:class: dropdown

The residuals are centered around zero, and the residuals vs fitted plot does not show a severe curve or funnel shape. The Q-Q plot is not perfect, but with only eight observations there is limited evidence either way. The residuals by week should be checked because the data are ordered in time. Overall, the model is reasonable for a first simple-regression example and for interpolation inside the observed temperature range, but uncertainty should be reported with prediction intervals.
```

## What to remember

Diagnostics do not ask whether the model is "true." They ask whether the model is useful enough for the purpose at hand and whether its limitations are visible.